# Day 3, Session 3 -- Capstone: Multi-Agent Consulting System

You are the McKinsey engineer building the demo. A Partner wants a working prototype: an **Engagement Manager agent** takes a complex consulting question, structures a plan, and hands it off to a **Specialist agent** who answers using the knowledge base you built in Session 1. Optionally, a **Quality Reviewer** evaluates the answer before it reaches the Partner.

Four milestones. One working system. Two-minute demo at the end.

---
## Capstone Goal

This is the **culmination of the entire 3-day program**. You will build a working multi-agent consulting system that demonstrates every major concept covered across all three days:

| Day | Session | Concept Applied Here |
|-----|---------|---------------------|
| Day 1 | Session 1 | OpenAI API calls, model selection, temperature control |
| Day 1 | Session 2 | Prompt engineering (system prompts, personas, structured instructions) |
| Day 1 | Session 3 | Evaluation criteria (relevance, evidence quality, actionability) |
| Day 2 | Session 1 | Structured outputs / JSON mode |
| Day 2 | Session 2 | LangChain tools (@tool decorator, tool calling) |
| Day 2 | Session 3 | LangGraph orchestration (StateGraph, conditional edges, cycles) |
| Day 2 | Session 4 | Multi-agent patterns (supervisor, handoffs, shared state) |
| Day 3 | Session 1 | RAG retrieval (embeddings, ChromaDB, citation) |
| Day 3 | Session 2 | Production patterns (error handling, scaling considerations) |

**The System You Will Build:**

1. **Engagement Manager Agent** -- Plans the work (JSON structured output)
2. **Specialist Agent with RAG** -- Retrieves knowledge and produces cited answers
3. **Quality Review Agent** (Optional) -- Scores and gates the output
4. **LangGraph Pipeline** (Optional/Intermediate) -- Wires everything into an orchestrated graph with conditional routing

By the end of this session, you will have a system that a Partner could type a question into and receive a structured, evidence-backed, quality-checked consulting answer.

---
## Your Role

You are the engineer. The Partner and the engagement team are the users -- they will type questions and read the output your system produces. Your engineering decisions: how the EM structures its plan, what persona the specialist uses, and how the RAG tool retrieves and formats context.

The corpus is the same consulting knowledge base you indexed in D3S1 (frameworks, playbooks, industry reports). The specialist agent calls it as a tool to ground its answers in real content.

---
## Setup

We will set up the environment in three steps:
1. Install dependencies and import libraries
2. Define the consulting knowledge base
3. Index documents in ChromaDB for retrieval

### Step 1: Install Dependencies and Imports

We need the OpenAI SDK for LLM calls and embeddings, LangChain/LangGraph for orchestration, and ChromaDB as our vector store.

In [ ]:
!pip install -q openai langchain langchain-openai langchain-core langgraph chromadb python-dotenv

In [ ]:
import os
import json
import numpy as np
from typing import TypedDict, Annotated, Literal
from dotenv import load_dotenv
load_dotenv()

import openai
import chromadb
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

# Initialize clients
client = openai.OpenAI()
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=0)

print("All imports successful. Clients initialized.")

### Step 2: Define the Consulting Knowledge Base

This is the same corpus from Day 3 Session 1. It contains McKinsey frameworks, playbooks, and reports that the specialist agent will retrieve from. In a production system, this would be thousands of documents -- here we use a representative sample.

In [ ]:
# ============================================================
# Consulting Knowledge Base (same corpus as D3S1)
# ============================================================
knowledge_docs = [
    {
        "text": "McKinsey's post-merger integration (PMI) framework emphasizes three pillars: capturing synergies within the first 100 days, aligning organizational culture through structured leadership workshops, and establishing a clean-room integration management office (IMO). Research across 200+ acquisitions shows that companies capturing 80% of synergies in year one outperform peers by 15% in total shareholder returns.",
        "source": "pmi_framework_2024.md",
        "metadata": {"practice_area": "M&A", "industry": "Cross-industry"}
    },
    {
        "text": "Digital transformation in manufacturing requires a phased approach: Phase 1 -- Assess digital maturity across the value chain using McKinsey's Digital Quotient (DQ) diagnostic. Phase 2 -- Prioritize use cases by impact and feasibility (typically predictive maintenance, demand sensing, and quality analytics yield highest ROI). Phase 3 -- Build a scalable data platform and upskill the workforce.",
        "source": "digital_manufacturing_playbook.md",
        "metadata": {"practice_area": "Digital", "industry": "Manufacturing"}
    },
    {
        "text": "Operations excellence programs typically deliver 15-25% cost reduction in the first 18 months. McKinsey's approach begins with a diagnostic spanning procurement, production, and logistics. Lean management principles are combined with advanced analytics to identify waste. A critical success factor is embedding a performance management infrastructure -- daily management systems, KPI cascades, and capability-building academies.",
        "source": "ops_excellence_guide.md",
        "metadata": {"practice_area": "Operations", "industry": "Cross-industry"}
    },
    {
        "text": "CEO briefing templates should follow the Situation-Complication-Resolution (SCR) structure. Open with a one-sentence situation framing the strategic context. Introduce the complication -- the specific challenge or opportunity requiring action. Close with the resolution -- McKinsey's recommended path forward with 2-3 actionable next steps. Keep the core narrative to 5-7 pages maximum.",
        "source": "ceo_briefing_template.md",
        "metadata": {"practice_area": "Leadership", "industry": "Cross-industry"}
    },
    {
        "text": "Supply chain resilience has become a board-level priority following recent global disruptions. McKinsey's supply chain diagnostic evaluates five dimensions: supplier diversification, inventory buffering strategy, logistics network flexibility, digital visibility (control tower maturity), and workforce agility. Best-in-class companies maintain dual sourcing for critical components and hold 4-6 weeks of safety stock for high-risk SKUs.",
        "source": "supply_chain_resilience_report.md",
        "metadata": {"practice_area": "Operations", "industry": "Manufacturing"}
    }
]

print(f"Knowledge base defined: {len(knowledge_docs)} documents")
for doc in knowledge_docs:
    print(f"  - {doc['source']} ({doc['metadata']['practice_area']}, {doc['metadata']['industry']})")

### Step 3: Index in ChromaDB

We embed each document using OpenAI's text-embedding-3-small model and store them in a ChromaDB collection. This creates the vector index that our RAG tool will query in Milestone 2.

In [ ]:
# Create ChromaDB collection and index documents
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("capstone_knowledge")

# Generate embeddings
texts = [d["text"] for d in knowledge_docs]
emb_response = client.embeddings.create(model="text-embedding-3-small", input=texts)
embeddings = [item.embedding for item in emb_response.data]

# Add to collection
collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=[f"doc_{i}" for i in range(len(texts))],
    metadatas=[{**d["metadata"], "source": d["source"]} for d in knowledge_docs]
)

print(f"Indexed {len(texts)} documents in ChromaDB.")
print(f"Collection count: {collection.count()}")
print("\nSetup complete! Ready to build the multi-agent system.")

---
## Milestone 1 (Easy): Engagement Manager Agent

The Engagement Manager (EM) is the **planning layer** of our system. In a real McKinsey engagement, the EM takes a complex client question, decomposes it into workstreams, and assigns specialists. Our EM agent does the same thing programmatically.

**What the EM does:**
- Accepts a consulting question (e.g., PE due diligence, market entry, digital transformation)
- Returns a structured JSON plan naming the workstream, the specialist to assign, and the hypothesis to test
- Does NOT answer the question itself -- it only plans

**Why JSON mode?** The EM's output feeds directly into Milestone 2 as structured input. Without guaranteed JSON, the pipeline would break on malformed output. This is the same pattern you learned in Day 2, Session 1 (Structured Outputs).

**Requirements:**
- Use `response_format={"type": "json_object"}` for reliable structured output
- Return JSON with exactly these keys: `workstream`, `specialist`, `hypothesis`
- Available specialists: `strategy_analyst`, `financial_analyst`, `operations_expert`, `industry_researcher`

### Step 1.1: Define the EM System Prompt

The system prompt must instruct the model to:
- Act as a McKinsey Engagement Manager
- Decompose the question into a workstream
- Select the right specialist
- Formulate a testable hypothesis
- Return ONLY valid JSON

In [ ]:
# Milestone 1, Step 1: Define the EM system prompt
#
# TODO 1: Write a system prompt that instructs the LLM to act as a
#          McKinsey Engagement Manager. The prompt should:
#          - Establish the EM persona (senior, experienced, analytical)
#          - List the available specialists to choose from
#          - Specify the exact JSON output format required
#          - Emphasize planning only, not answering

EM_SYSTEM_PROMPT = """
# TODO 1: Define the system prompt here
# Your prompt should instruct the model to return JSON with:
#   - "workstream": a clear description of what to investigate
#   - "specialist": one of [strategy_analyst, financial_analyst, operations_expert, industry_researcher]
#   - "hypothesis": what the specialist should test/validate
"""

print("EM system prompt defined.")
print(f"Prompt length: {len(EM_SYSTEM_PROMPT)} characters")

### Step 1.2: Build the EM Function with JSON Mode

Now wire up the API call. Use `response_format={"type": "json_object"}` to guarantee valid JSON output. Parse the response and return a Python dictionary.

In [ ]:
# Milestone 1, Step 2: Build the EM function
#
# TODO 2: Implement the engagement_manager function that:
#          - Calls the OpenAI API with the EM system prompt
#          - Uses response_format={"type": "json_object"} for structured output
#          - Uses temperature=0 for deterministic planning
#          - Parses the JSON response into a Python dict
#          - Returns the dict with keys: workstream, specialist, hypothesis

def engagement_manager(question: str) -> dict:
    """Takes a consulting question, returns a structured engagement plan as JSON."""
    
    # TODO 2a: Make the API call with JSON mode
    # response = client.chat.completions.create(
    #     model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    #     temperature=0,
    #     response_format={"type": "json_object"},
    #     messages=[...]
    # )
    
    # TODO 2b: Parse the JSON response
    # plan = json.loads(response.choices[0].message.content)
    
    # TODO 2c: Return the parsed plan
    pass


print("engagement_manager() function defined.")

### Step 1.3: Test the Engagement Manager

Run a test question through the EM and verify the output structure.

In [ ]:
# Milestone 1, Step 3: Test the EM
#
# TODO 3: Uncomment and run the test below. Verify that:
#          - The output is valid JSON
#          - It contains all three required keys
#          - The specialist is one of the valid options
#          - The hypothesis is testable and specific

# test_question = (
#     "Our PE client is evaluating a $500M acquisition of a healthcare IT platform. "
#     "Assess the target's competitive positioning and growth trajectory."
# )
#
# plan = engagement_manager(test_question)
# print("=== EM Plan ===")
# print(json.dumps(plan, indent=2))
# print()
#
# # Validation
# assert "workstream" in plan, "Missing 'workstream' key"
# assert "specialist" in plan, "Missing 'specialist' key"
# assert "hypothesis" in plan, "Missing 'hypothesis' key"
# assert plan["specialist"] in ["strategy_analyst", "financial_analyst", "operations_expert", "industry_researcher"], \
#     f"Invalid specialist: {plan['specialist']}"
# print("All assertions passed!")

### Milestone 1 Hints

<details>
<summary><strong>Hint 1: System prompt structure</strong></summary>

A good EM system prompt follows this pattern:
```
You are a senior McKinsey Engagement Manager. Your role is to...

Available specialists:
- strategy_analyst: ...
- financial_analyst: ...
- operations_expert: ...
- industry_researcher: ...

You must respond with ONLY a JSON object containing:
- "workstream": ...
- "specialist": ...
- "hypothesis": ...
```
</details>

<details>
<summary><strong>Hint 2: API call structure</strong></summary>

```python
response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    temperature=0,
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": EM_SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]
)
```
</details>

<details>
<summary><strong>Hint 3: Complete solution</strong></summary>

```python
def engagement_manager(question: str) -> dict:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": EM_SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ]
    )
    plan = json.loads(response.choices[0].message.content)
    return plan
```
</details>

**Expected output shape** (your exact wording will vary):
```json
{
  "workstream": "Competitive positioning and growth trajectory analysis of the healthcare IT target",
  "specialist": "strategy_analyst",
  "hypothesis": "The target holds a defensible market position with above-market growth driven by ..."
}
```

---
## Milestone 2 (Easy): Specialist Agent with RAG Tool

The specialist receives the EM's assignment, retrieves relevant knowledge from the consulting corpus, and produces a grounded, cited answer.

**Why citations?** In consulting, every claim must be traceable to evidence. The specialist cites sources so the Partner can verify the analysis -- just like a real engagement. An uncited claim is an unverified claim, and in a $500M deal, that is unacceptable.

This milestone uses:
- The `@tool` decorator from LangChain (Day 2, Session 2)
- Embeddings + vector search (Day 3, Session 1)
- Prompt engineering for the specialist persona (Day 1, Session 2)

**Requirements:**
- Define a RAG retrieval tool using the `@tool` decorator
- The tool searches ChromaDB and returns relevant chunks with source metadata
- The specialist receives workstream + hypothesis, calls RAG, writes a cited answer
- Every factual claim includes a citation: `[Source: filename.md]`

### Step 2.1: Define the RAG Tool

The RAG tool takes a query string, embeds it, searches ChromaDB, and returns formatted results with source attribution.

In [ ]:
# Milestone 2, Step 1: Define the RAG retrieval tool
#
# TODO 1: Implement the search_knowledge_base tool that:
#          - Embeds the query using client.embeddings.create()
#          - Searches collection.query(query_embeddings=[...], n_results=3)
#          - Formats each result as "[Source: filename] chunk_text"
#          - Returns all formatted results joined by newlines

@tool
def search_knowledge_base(query: str) -> str:
    """Search the McKinsey consulting knowledge base for relevant frameworks, playbooks, and reports."""
    
    # TODO 1a: Embed the query
    # query_emb = client.embeddings.create(
    #     model="text-embedding-3-small", input=[query]
    # ).data[0].embedding
    
    # TODO 1b: Search ChromaDB
    # results = collection.query(
    #     query_embeddings=[query_emb],
    #     n_results=3
    # )
    
    # TODO 1c: Format results with source citations
    # formatted = []
    # for doc, metadata in zip(results["documents"][0], results["metadatas"][0]):
    #     formatted.append(f"[Source: {metadata['source']}] {doc}")
    # return "\n\n".join(formatted)
    
    pass


print("search_knowledge_base tool defined.")

### Step 2.2: Test the RAG Tool in Isolation

Before building the specialist, verify the tool works correctly on its own.

In [ ]:
# Test the RAG tool directly
#
# TODO 2: Uncomment and verify the tool returns relevant, cited chunks

# test_results = search_knowledge_base.invoke("post-merger integration synergies")
# print("=== RAG Tool Test ===")
# print(test_results)
# print()
# assert "[Source:" in test_results, "Missing source citations in RAG output"
# print("RAG tool working correctly!")

### Step 2.3: Build the Specialist Agent Function

The specialist takes the EM's workstream and hypothesis, calls the RAG tool to get evidence, then produces a structured answer with citations.

In [ ]:
# Milestone 2, Step 3: Build the specialist agent
#
# TODO 3: Implement the specialist_agent function that:
#          - Constructs a search query from the workstream and hypothesis
#          - Calls search_knowledge_base to get relevant context
#          - Sends context + assignment to the LLM with a specialist system prompt
#          - Returns the LLM's cited answer

SPECIALIST_SYSTEM_PROMPT = """
# TODO 3a: Define the specialist system prompt
# The prompt should instruct the model to:
# - Act as a McKinsey consulting specialist
# - Use ONLY the provided context to answer
# - Cite every claim using [Source: filename.md] format
# - Structure the answer with clear sections
# - Be actionable and specific
"""


def specialist_agent(workstream: str, hypothesis: str) -> str:
    """A consulting specialist that uses RAG to answer workstream questions with citations."""
    
    # TODO 3b: Call the RAG tool to retrieve relevant context
    # search_query = f"{workstream} {hypothesis}"
    # context = search_knowledge_base.invoke(search_query)
    
    # TODO 3c: Build the user message with context and assignment
    # user_message = f"""
    # ## Assignment
    # Workstream: {workstream}
    # Hypothesis to test: {hypothesis}
    #
    # ## Retrieved Evidence
    # {context}
    #
    # Provide your analysis with citations.
    # """
    
    # TODO 3d: Call the LLM and return the response
    # response = client.chat.completions.create(
    #     model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    #     temperature=0.2,
    #     messages=[...]
    # )
    # return response.choices[0].message.content
    
    pass


print("specialist_agent() function defined.")

### Step 2.4: End-to-End Test (EM + Specialist)

Run a consulting question through both agents: EM plans, then specialist retrieves and answers.

In [ ]:
# Milestone 2, Step 4: End-to-end test
#
# TODO 4: Uncomment and run the full pipeline

# test_question = (
#     "Our PE client is evaluating a $500M acquisition of a healthcare IT platform. "
#     "Assess the target's competitive positioning and growth trajectory."
# )
#
# # Step 1: EM plans
# plan = engagement_manager(test_question)
# print("=== EM Plan ===")
# print(json.dumps(plan, indent=2))
# print()
#
# # Step 2: Specialist answers
# answer = specialist_agent(plan["workstream"], plan["hypothesis"])
# print("=== Specialist Answer ===")
# print(answer)
# print()
#
# # Verify citations exist
# assert "[Source:" in answer, "Specialist answer missing citations!"
# print("\nEnd-to-end pipeline working!")

### Milestone 2 Hints

<details>
<summary><strong>Hint 1: RAG tool implementation</strong></summary>

```python
@tool
def search_knowledge_base(query: str) -> str:
    """Search the McKinsey consulting knowledge base."""
    query_emb = client.embeddings.create(
        model="text-embedding-3-small", input=[query]
    ).data[0].embedding
    
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=3
    )
    
    formatted = []
    for doc, metadata in zip(results["documents"][0], results["metadatas"][0]):
        formatted.append(f"[Source: {metadata['source']}] {doc}")
    return "\n\n".join(formatted)
```
</details>

<details>
<summary><strong>Hint 2: Specialist system prompt</strong></summary>

```
You are a senior McKinsey consulting specialist. You produce rigorous,
evidence-based analysis for engagement teams.

Rules:
- Use ONLY the provided context to support your analysis
- Cite every factual claim using [Source: filename.md] format
- Structure your answer with clear headers
- Be specific and actionable
- If the context does not support the hypothesis, say so explicitly
```
</details>

<details>
<summary><strong>Hint 3: Complete specialist function</strong></summary>

```python
def specialist_agent(workstream: str, hypothesis: str) -> str:
    search_query = f"{workstream} {hypothesis}"
    context = search_knowledge_base.invoke(search_query)
    
    user_message = f"""## Assignment
Workstream: {workstream}
Hypothesis to test: {hypothesis}

## Retrieved Evidence
{context}

Provide your analysis with citations."""
    
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        temperature=0.2,
        messages=[
            {"role": "system", "content": SPECIALIST_SYSTEM_PROMPT},
            {"role": "user", "content": user_message}
        ]
    )
    return response.choices[0].message.content
```
</details>

**Expected output shape:**

The specialist's answer should:
- Reference content from the knowledge base (PMI framework, operations excellence, etc.)
- Include inline citations like `[Source: pmi_framework_2024.md]`
- Be structured and actionable -- the Partner will read this

---
## Milestone 3 (Intermediate): Quality Review Agent

In a real engagement, work products go through quality review before reaching the client. A senior colleague reviews the analysis for rigor, evidence quality, and actionability.

This milestone builds a **QA agent** that evaluates the specialist's answer on three criteria and can send it back for improvement. This references:
- **Day 1, Session 3** (Model Evaluation) -- scoring criteria and rubrics
- **Day 2, Session 3** (LangGraph) -- conditional routing and cycles

**What the QA Agent does:**
1. Takes the specialist's answer + the original workstream/hypothesis
2. Scores on 3 criteria (1-5 scale each):
   - **Relevance**: Does the answer address the workstream and hypothesis?
   - **Evidence Quality**: Are claims supported by cited sources?
   - **Actionability**: Could a Partner act on this analysis?
3. Returns JSON with scores, reasoning, and a pass/fail verdict
4. If FAIL: provides improvement instructions and loops back to specialist (max 2 iterations)

**Pass threshold:** Average score >= 3.5

### Step 3.1: Define the QA Agent

In [ ]:
# Milestone 3, Step 1: Define the Quality Review agent
#
# TODO 1: Implement the quality_review_agent function that:
#          - Takes the specialist's answer, workstream, and hypothesis
#          - Calls the LLM with a QA system prompt and JSON mode
#          - Returns a dict with: relevance_score, evidence_score,
#            actionability_score, reasoning, verdict ("pass"/"fail"),
#            and improvement_instructions (if fail)

QA_SYSTEM_PROMPT = """
# TODO 1a: Define the QA system prompt
# The prompt should instruct the model to:
# - Act as a senior McKinsey quality reviewer
# - Score the answer on relevance (1-5), evidence_quality (1-5), actionability (1-5)
# - Provide brief reasoning for each score
# - Give a verdict: "pass" if average >= 3.5, otherwise "fail"
# - If fail, provide specific improvement_instructions
# - Return ONLY valid JSON
"""


def quality_review_agent(answer: str, workstream: str, hypothesis: str) -> dict:
    """Reviews specialist output and returns scores + pass/fail verdict."""
    
    # TODO 1b: Build the user message with the answer to review
    # user_message = f"""
    # ## Original Assignment
    # Workstream: {workstream}
    # Hypothesis: {hypothesis}
    #
    # ## Answer to Review
    # {answer}
    #
    # Score this answer and provide your verdict.
    # """
    
    # TODO 1c: Call the API with JSON mode
    # response = client.chat.completions.create(
    #     model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    #     temperature=0,
    #     response_format={"type": "json_object"},
    #     messages=[...]
    # )
    
    # TODO 1d: Parse and return the review
    # review = json.loads(response.choices[0].message.content)
    # return review
    
    pass


print("quality_review_agent() function defined.")

### Step 3.2: Build the Review Loop

If the QA agent returns "fail", loop back to the specialist with improvement instructions. Cap at 2 iterations to avoid infinite loops.

In [ ]:
# Milestone 3, Step 2: Build the review loop
#
# TODO 2: Implement the review_loop function that:
#          - Gets the specialist's answer
#          - Sends it to the QA agent
#          - If "pass", returns the answer
#          - If "fail", calls the specialist again with improvement instructions
#          - Max 2 iterations, then returns whatever we have

def review_loop(workstream: str, hypothesis: str, max_iterations: int = 2) -> dict:
    """Runs specialist + QA review loop with max iterations."""
    
    # TODO 2a: Get initial specialist answer
    # answer = specialist_agent(workstream, hypothesis)
    
    # TODO 2b: Review loop
    # for iteration in range(max_iterations):
    #     review = quality_review_agent(answer, workstream, hypothesis)
    #     print(f"  Iteration {iteration + 1}: Verdict = {review['verdict']}")
    #
    #     if review["verdict"] == "pass":
    #         return {"answer": answer, "review": review, "iterations": iteration + 1}
    #
    #     # Fail -- get improvement instructions and retry
    #     improvement = review.get("improvement_instructions", "Improve the answer.")
    #     # Call specialist again with additional instructions
    #     answer = specialist_agent(
    #         workstream,
    #         f"{hypothesis}\n\nIMPROVEMENT NEEDED: {improvement}"
    #     )
    
    # TODO 2c: Return final result after max iterations
    # return {"answer": answer, "review": review, "iterations": max_iterations}
    
    pass


print("review_loop() function defined.")

### Step 3.3: Test the Review Loop

In [ ]:
# Milestone 3, Step 3: Test the full review loop
#
# TODO 3: Uncomment and test

# test_question = (
#     "How should we approach post-merger integration for a mid-size "
#     "manufacturing acquisition to capture synergies quickly?"
# )
#
# plan = engagement_manager(test_question)
# print("=== EM Plan ===")
# print(json.dumps(plan, indent=2))
# print()
#
# print("=== Running Review Loop ===")
# result = review_loop(plan["workstream"], plan["hypothesis"])
# print()
# print(f"=== Final Answer (after {result['iterations']} iteration(s)) ===")
# print(result["answer"])
# print()
# print("=== QA Scores ===")
# review = result["review"]
# print(f"  Relevance:     {review.get('relevance_score', 'N/A')}/5")
# print(f"  Evidence:      {review.get('evidence_score', 'N/A')}/5")
# print(f"  Actionability: {review.get('actionability_score', 'N/A')}/5")
# print(f"  Verdict:       {review.get('verdict', 'N/A')}")

### Milestone 3 Hints

<details>
<summary><strong>Hint 1: QA system prompt structure</strong></summary>

```
You are a senior McKinsey quality reviewer. You evaluate consulting work products
for rigor and client-readiness.

Score the answer on these criteria (1-5 each):
1. relevance_score: Does it address the workstream and hypothesis?
2. evidence_score: Are claims supported by cited sources?
3. actionability_score: Could a Partner act on this?

Return JSON with:
- relevance_score, evidence_score, actionability_score (integers 1-5)
- reasoning (brief explanation for each score)
- verdict: "pass" if average >= 3.5, else "fail"
- improvement_instructions: (only if fail) specific guidance to improve
```
</details>

<details>
<summary><strong>Hint 2: Expected QA output shape</strong></summary>

```json
{
  "relevance_score": 4,
  "evidence_score": 3,
  "actionability_score": 4,
  "reasoning": "The answer addresses the workstream well but could cite more sources...",
  "verdict": "pass",
  "improvement_instructions": null
}
```
</details>

---
## Milestone 4 (Intermediate): LangGraph Multi-Agent Pipeline

This is the **ultimate capstone** -- wire Milestones 1-3 together as a LangGraph StateGraph with conditional routing.

**This milestone brings together:**
- LangGraph orchestration (Day 2, Session 3)
- Multi-agent patterns (Day 2, Session 4)
- RAG retrieval (Day 3, Session 1)
- Evaluation criteria (Day 1, Session 3)
- Structured outputs (Day 2, Session 1)
- Prompt engineering (Day 1, Session 2)

**Pipeline Architecture:**
```
[Input Question]
       |
       v
  [Node 1: EM Plans]
       |
       v
  [Node 2: Specialist Answers with RAG]
       |
       v
  [Node 3: QA Reviews]
       |
       v
  [Conditional Edge]
     /       \
  PASS      FAIL (& iterations < max)
    |          \
    v           v
 [Output]   [Back to Node 2 with feedback]
```

**State schema:** The graph state carries the question, plan, answer, review, and iteration count between nodes.

### Step 4.1: Define the Graph State

In [ ]:
# Milestone 4, Step 1: Define the state schema
#
# TODO 1: Define a TypedDict that carries all data between nodes

from langgraph.graph import StateGraph, END

class ConsultingState(TypedDict):
    """State shared across all nodes in the consulting pipeline."""
    # TODO 1: Add the required fields
    question: str              # The original consulting question
    plan: dict                 # EM's JSON plan (workstream, specialist, hypothesis)
    answer: str                # Specialist's cited answer
    review: dict               # QA agent's scores and verdict
    iteration: int             # Current iteration count
    max_iterations: int        # Maximum allowed iterations
    improvement_feedback: str  # Feedback from QA if answer fails


print("ConsultingState defined.")

### Step 4.2: Define Node Functions

Each node takes state, does its work, and returns updated state fields.

In [ ]:
# Milestone 4, Step 2: Define node functions
#
# TODO 2: Implement each node function. Each takes state and returns
#          a dict of fields to update in state.

def em_node(state: ConsultingState) -> dict:
    """Node 1: Engagement Manager plans the work."""
    # TODO 2a: Call engagement_manager with state["question"]
    # Return {"plan": plan}
    pass


def specialist_node(state: ConsultingState) -> dict:
    """Node 2: Specialist retrieves knowledge and answers."""
    # TODO 2b: Get workstream and hypothesis from state["plan"]
    # If there's improvement_feedback, append it to the hypothesis
    # Call specialist_agent and increment iteration
    # Return {"answer": answer, "iteration": state["iteration"] + 1}
    pass


def qa_node(state: ConsultingState) -> dict:
    """Node 3: Quality reviewer scores the answer."""
    # TODO 2c: Call quality_review_agent with answer, workstream, hypothesis
    # Extract improvement_instructions if verdict is fail
    # Return {"review": review, "improvement_feedback": instructions}
    pass


print("Node functions defined.")

### Step 4.3: Define the Conditional Edge

The conditional edge after QA decides whether to output the result or loop back to the specialist.

In [ ]:
# Milestone 4, Step 3: Define the routing function
#
# TODO 3: Implement the routing logic

def should_continue(state: ConsultingState) -> Literal["specialist", "end"]:
    """Decide whether to loop back to specialist or finish."""
    # TODO 3: Return "end" if:
    #   - review verdict is "pass", OR
    #   - iteration >= max_iterations
    # Otherwise return "specialist" to retry
    pass


print("Routing function defined.")

### Step 4.4: Build and Compile the Graph

In [ ]:
# Milestone 4, Step 4: Assemble the StateGraph
#
# TODO 4: Build the graph with nodes and edges

# TODO 4a: Create the graph
# graph = StateGraph(ConsultingState)

# TODO 4b: Add nodes
# graph.add_node("em", em_node)
# graph.add_node("specialist", specialist_node)
# graph.add_node("qa", qa_node)

# TODO 4c: Add edges
# graph.set_entry_point("em")
# graph.add_edge("em", "specialist")
# graph.add_edge("specialist", "qa")

# TODO 4d: Add conditional edge from qa
# graph.add_conditional_edges(
#     "qa",
#     should_continue,
#     {
#         "specialist": "specialist",
#         "end": END
#     }
# )

# TODO 4e: Compile
# consulting_pipeline = graph.compile()

# print("LangGraph pipeline compiled!")
# print("Nodes:", list(consulting_pipeline.nodes.keys()))

### Step 4.5: Run the Full Pipeline

In [ ]:
# Milestone 4, Step 5: Execute the pipeline
#
# TODO 5: Run a question through the full graph

# initial_state = {
#     "question": (
#         "Our PE client is evaluating a $500M acquisition of a healthcare IT platform. "
#         "Assess the target's competitive positioning and growth trajectory."
#     ),
#     "plan": {},
#     "answer": "",
#     "review": {},
#     "iteration": 0,
#     "max_iterations": 2,
#     "improvement_feedback": ""
# }
#
# print("=== Running Full LangGraph Pipeline ===")
# print(f"Question: {initial_state['question']}")
# print()
#
# # Execute
# final_state = consulting_pipeline.invoke(initial_state)
#
# # Display results
# print("=" * 60)
# print("=== FINAL RESULTS ===")
# print("=" * 60)
# print()
# print("--- EM Plan ---")
# print(json.dumps(final_state["plan"], indent=2))
# print()
# print("--- Specialist Answer ---")
# print(final_state["answer"])
# print()
# print("--- QA Review ---")
# print(json.dumps(final_state["review"], indent=2))
# print()
# print(f"Total iterations: {final_state['iteration']}")
# print(f"Final verdict: {final_state['review'].get('verdict', 'N/A')}")

### Milestone 4 Hints

<details>
<summary><strong>Hint 1: Node function pattern</strong></summary>

Each node function takes state and returns only the fields it updates:

```python
def em_node(state: ConsultingState) -> dict:
    plan = engagement_manager(state["question"])
    return {"plan": plan}
```
</details>

<details>
<summary><strong>Hint 2: Conditional edge routing</strong></summary>

```python
def should_continue(state: ConsultingState) -> Literal["specialist", "end"]:
    if state["review"].get("verdict") == "pass":
        return "end"
    if state["iteration"] >= state["max_iterations"]:
        return "end"
    return "specialist"
```
</details>

---
## Presentation -- 2 Minutes

Demo your working system. Run a consulting question end-to-end.

### What to Demonstrate

- **The EM planning**: Show a question going in and a structured JSON plan coming out
- **The specialist answering**: Show the RAG retrieval providing evidence, and the final cited answer
- **The quality gate** (if you built Milestone 3): Show the scoring and pass/fail mechanism
- **The full pipeline** (if you built Milestone 4): Show the graph executing end-to-end, including any retry loops

### Design Decisions to Discuss

Name **one or two design decisions** you made. Examples:

- "I used temperature=0 for the EM because the plan needs to be deterministic -- you do not want the same question routed differently each time."
- "I set k=3 for retrieval because more chunks diluted answer quality with irrelevant content."
- "I set the pass threshold at 3.5 because lower thresholds let weak answers through, and higher ones caused too many retries."
- "I chose to append improvement feedback to the hypothesis rather than the system prompt because it keeps the specialist's core behavior stable."
- "I capped iterations at 2 because in testing, answers that failed twice rarely improved on a third attempt."

No slides. Just the running notebook and clear articulation of tradeoffs.

### Reflection Questions

Consider these questions for the debrief discussion:

1. **Scaling**: What would you change if the knowledge base had 10,000 documents instead of 5? Would you change the embedding model, the chunking strategy, or the retrieval approach?

2. **Semantic Caching**: If the same types of questions come in repeatedly (e.g., "assess competitive positioning" for different targets), how would you add semantic caching to avoid redundant LLM calls?

3. **Production Metrics**: What metrics would you track in production? Consider latency per node, QA pass rates, retrieval relevance scores, token costs per query, and user satisfaction.

4. **Error Handling**: What happens if the EM returns an invalid specialist name? If ChromaDB is down? If the LLM returns malformed JSON despite JSON mode? How would you make this system resilient?

5. **Agent Autonomy**: Should the specialist be able to ask clarifying questions back to the EM? Should the QA agent be able to request additional retrieval? What are the tradeoffs of more autonomous agents vs. a fixed pipeline?

6. **Human-in-the-Loop**: Where would you add human checkpoints in a production version? After the EM plans? Before the final answer goes to the client? How do you balance automation with oversight?

In [ ]:
# (Optional) Run your final demo here -- a single cell that executes
# the full pipeline for your presentation

# demo_question = "Your consulting question here"
# ...